In [3]:
"""
Project 01: Credit Risk Prediction
Step 01: Exploratory Data Analysis
"""

'\nProject 01: Credit Risk Prediction\nStep 01: Exploratory Data Analysis\n'

In [3]:
import warnings

In [4]:
warnings.filterwarnings("ignore")

In [5]:
import pandas as pd
import numpy as np
import matplotlib

In [6]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
import time
from tqdm import tqdm

In [7]:
from path_setup import BASE, RAW, PROCESSED, MODELS, CHARTS, load_raw_csv

df = load_raw_csv("application_train.csv")
df.head()


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# RAW = os.path.join(BASE, "data", "raw")
# PROCESSED = os.path.join(BASE, "data", "processed")
# MODELS = os.path.join(BASE, "models")
# # CHARTS = os.path.join(BASE, "charts")

In [10]:
# os.makedirs(PROCESSED, exist_ok=True)
# os.makedirs(MODELS, exist_ok=True)
# os.makedirs(CHARTS, exist_ok=True)

In [11]:
print("=" * 60)
print("STEP 01: Exploratory Data Analysis")
print("=" * 60)
start_time = time.time()

STEP 01: Exploratory Data Analysis


In [12]:
# ── Load Data ──────────────────────────────────────────────
print("\nLoading application_train.csv ...")
df = pd.read_csv(os.path.join(RAW, "application_train.csv"))
print(f"Shape:  {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Default rate: {df['TARGET'].mean():.2%}")


Loading application_train.csv ...
Shape:  23,895 rows × 122 columns
Memory: 43.7 MB
Default rate: 8.01%


In [13]:
# ── 1. Target Variable Analysis ──────────────────────────
print("\n[1/5] Target variable analysis ...")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))


[1/5] Target variable analysis ...


In [14]:
counts = df["TARGET"].value_counts()
axes[0].bar(
    ["Repaid (0)", "Default (1)"],
    counts.values,
    color=["#6C63FF", "#FF6584"],
    alpha=0.9,
)
for i, v in enumerate(counts.values):
    axes[0].text(
        i, v + 1000, f"{v:,}\n({v / len(df) * 100:.1f}%)", ha="center", fontsize=11
    )
axes[0].set_title("Class Distribution", fontweight="bold")
axes[0].set_ylabel("Count")

Text(0, 0.5, 'Count')

In [15]:
naive_acc = counts[0] / counts.sum()
axes[1].bar(
    ["Naive Model\n(always 0)", "Smart Model\n(AUROC ~0.79)"],
    [naive_acc, 0.79],
    color=["#888", "#6C63FF"],
    alpha=0.85,
)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Score")
axes[1].set_title("Why Accuracy is Wrong Metric\nUse AUROC instead!", fontweight="bold")
for i, v in enumerate([naive_acc, 0.79]):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=12, fontweight="bold")

In [16]:
for t, color, label in [(0, "#6C63FF", "Repaid"), (1, "#FF6584", "Default")]:
    ages = -df[df["TARGET"] == t]["DAYS_BIRTH"] / 365
    axes[2].hist(ages, bins=40, alpha=0.6, color=color, label=label, density=True)
axes[2].set_xlabel("Age (years)")
axes[2].set_title("Age Distribution by Default Status", fontweight="bold")
axes[2].legend()

In [17]:
plt.suptitle("Target Variable Analysis", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "target_analysis.png"), dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: target_analysis.png")

  Saved: target_analysis.png


In [18]:
# ── 2. Missing Values ────────────────────────────────────
print("[2/5] Missing values analysis ...")
missing_rate = df.isnull().mean().sort_values(ascending=False)
high_missing = missing_rate[missing_rate > 0.3]

[2/5] Missing values analysis ...


In [19]:
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(
    high_missing.index[:25],
    high_missing.values[:25] * 100,
    color=["#FF6584" if v > 0.5 else "#6C63FF" for v in high_missing.values[:25]],
)
ax.axvline(50, color="orange", linestyle="--", lw=2, label="50% threshold (drop)")
ax.set_xlabel("Missing Rate (%)", fontsize=12)
ax.set_title("Columns with >30% Missing Values", fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "missing_values.png"), dpi=150, bbox_inches="tight")
plt.close()
print(f"  Columns > 50% missing (will drop): {(missing_rate > 0.5).sum()}")
print(f"  Columns > 0% missing:              {(missing_rate > 0).sum()}")
print(f"  Saved: missing_values.png")

  Columns > 50% missing (will drop): 40
  Columns > 0% missing:              85
  Saved: missing_values.png


In [20]:
# ── 3. Key Feature Distributions ─────────────────────────
print("[3/5] Key feature distributions ...")
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

[3/5] Key feature distributions ...


In [21]:
features = [
    ("EXT_SOURCE_1", "External Score 1"),
    ("EXT_SOURCE_2", "External Score 2"),
    ("EXT_SOURCE_3", "External Score 3"),
    ("AMT_INCOME_TOTAL", "Annual Income"),
    ("AMT_CREDIT", "Loan Amount"),
    ("AMT_ANNUITY", "Monthly EMI"),
]

In [22]:
for ax, (col, title) in zip(axes.flat, features):
    for t, color, label in [(0, "#6C63FF", "Repaid"), (1, "#FF6584", "Default")]:
        data = df[df["TARGET"] == t][col].dropna()
        if col in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]:
            data = data.clip(0, data.quantile(0.99))
        ax.hist(data, bins=40, alpha=0.6, color=color, label=label, density=True)
    corr = df[col].corr(df["TARGET"])
    ax.set_title(f"{title}\n(corr: {corr:.4f})", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)

In [23]:
plt.suptitle(
    "Key Feature Distributions by Default Status",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "feature_distributions.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print(f"  Saved: feature_distributions.png")

  Saved: feature_distributions.png


In [24]:
# ── 4. Correlation Analysis ──────────────────────────────
print("[4/5] Correlation analysis ...")
num_cols = df.select_dtypes(include="number").columns.tolist()
correlations = (
    df[num_cols]
    .corr()["TARGET"]
    .drop("TARGET")
    .abs()
    .sort_values(ascending=False)
    .head(20)
)

[4/5] Correlation analysis ...


In [25]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

In [26]:
colors = [
    "#FF6584" if correlations.iloc[i] > 0.1 else "#6C63FF"
    for i in range(len(correlations))
]
axes[0].barh(correlations.index, correlations.values, color=colors, alpha=0.85)
axes[0].set_xlabel("|Correlation with TARGET|")
axes[0].set_title("Top 20 Features by Correlation with Default", fontweight="bold")
axes[0].axvline(0.1, color="orange", linestyle="--", label="Strong signal threshold")
axes[0].legend()

In [27]:
key_cols = [
    "TARGET",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "AMT_CREDIT",
    "AMT_INCOME_TOTAL",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
]
corr_matrix = df[[c for c in key_cols if c in df.columns]].corr()
import seaborn as sns

In [28]:
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=axes[1],
    linewidths=0.5,
)
axes[1].set_title("Correlation Matrix — Key Features", fontweight="bold")

Text(0.5, 1.0, 'Correlation Matrix — Key Features')

In [29]:
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "correlation_analysis.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print(f"  Saved: correlation_analysis.png")

  Saved: correlation_analysis.png


In [30]:
# ── 5. Dataset Overview ──────────────────────────────────
print("[5/5] Dataset overview ...")
tables_files = {
    "application_train": "application_train.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "installments_payments": "installments_payments.csv",
    "POS_CASH_balance": "POS_CASH_balance.csv",
    "credit_card_balance": "credit_card_balance.csv",
    "application_test": "application_test.csv",
}

[5/5] Dataset overview ...


In [31]:
summary_rows = []
for name, fname in tqdm(tables_files.items(), desc="Scanning tables"):
    fpath = os.path.join(RAW, fname)
    if os.path.exists(fpath):
        tdf = pd.read_csv(fpath, nrows=0)
        row_count = sum(1 for _ in open(fpath)) - 1
        summary_rows.append(
            {
                "Table": name,
                "Rows": f"{row_count:,}",
                "Columns": len(tdf.columns),
            }
        )

Scanning tables: 100%|██████████| 8/8 [00:15<00:00,  1.98s/it]


In [32]:
summary = pd.DataFrame(summary_rows)
print("\nAll 8 Tables Summary:")
print(summary.to_string(index=False))


All 8 Tables Summary:
                Table       Rows  Columns
    application_train     23,895      122
               bureau  1,716,428       17
       bureau_balance 27,299,925        3
 previous_application  1,670,214       37
installments_payments 13,605,401        8
     POS_CASH_balance 10,001,358        8
  credit_card_balance  3,840,312       23
     application_test     48,744      121


In [33]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 01 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")


STEP 01 COMPLETE | Time: 86.3s
